# 🏦 Financial Agent Evaluation v4 — Azure AI Evaluation SDK

Uses the SDK’s evaluate() batch function to run **7 evaluators** across **36 test cases** (24 standard + 12 edge).

**Two-phase approach to avoid rate limiting:**
- **Phase 1**: Run the 3-agent pipeline sequentially with delays → save responses to JSONL
- **Phase 2**: Run evaluators in 2 batches on pre-computed data (quality + agentic)
- 3 agentic evaluators: TaskAdherence, IntentResolution, ResponseCompleteness
- Results uploaded to Foundry portal for run-over-run tracking (optional)

---
## Section 0 — Configuration

In [1]:
import os
import sys
from pathlib import Path

# Resolve repo root (works whether launched from repo root or notebooks/)
REPO_ROOT = Path(os.getcwd())
if (REPO_ROOT / "notebooks").exists():
    pass  # already at repo root
elif (REPO_ROOT.parent / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
from dotenv import load_dotenv
load_dotenv()  # Load .env file

# ── Foundry config ──
ENDPOINT        = os.environ["AZURE_AI_ENDPOINT"]
EVAL_ENDPOINT   = os.environ["AZURE_AI_EVAL_ENDPOINT"]
EVAL_DEPLOYMENT = "gpt-4o"   # ⬇️ CUSTOMIZE: your evaluation model deployment

AGENTS = {
    "summarizer":    "agent-summarizer",
    "clarification": "User-clarification-agent",  # ⬇️ CUSTOMIZE,
    "reporter":      "report-generator-agent",
}

EVAL_DATASET = "eval_dataset_v4.jsonl"
OUTPUT_DIR   = "eval_results_v4"

# Set to True to upload results to Foundry portal
UPLOAD_TO_FOUNDRY = False

print("✅ Config set")

✅ Config set


---
## Section 1 — Imports & Clients

In [2]:
import json, os, time
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import (
    evaluate,
    EvaluatorConfig,
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    TaskAdherenceEvaluator,
    IntentResolutionEvaluator,
    ResponseCompletenessEvaluator,
)

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=ENDPOINT, credential=credential)
oai = project_client.get_openai_client()

# Model config shared by all evaluators
eval_model_config = {
    "azure_endpoint": EVAL_ENDPOINT,
    "azure_deployment": EVAL_DEPLOYMENT,
}

print("✅ Clients ready")

✅ Clients ready


---
## Section 2 — Preview Dataset

In [3]:
preview_df = pd.read_json(EVAL_DATASET, lines=True)

std_count = len(preview_df[preview_df["category"] == "standard"])
edge_count = len(preview_df[preview_df["category"] == "edge"])

display(HTML(f"""
<div style='display:flex; gap:16px; margin:12px 0;'>
  <div style='flex:1; background:#2196F3; color:white; border-radius:10px; padding:16px; text-align:center;'>
    <div style='font-size:32px; font-weight:bold;'>{len(preview_df)}</div>
    <div>Total Test Cases</div>
  </div>
  <div style='flex:1; background:#4CAF50; color:white; border-radius:10px; padding:16px; text-align:center;'>
    <div style='font-size:32px; font-weight:bold;'>{std_count}</div>
    <div>Standard Q&A</div>
  </div>
  <div style='flex:1; background:#FF9800; color:white; border-radius:10px; padding:16px; text-align:center;'>
    <div style='font-size:32px; font-weight:bold;'>{edge_count}</div>
    <div>Edge Cases</div>
  </div>
</div>
"""))

# Edge case breakdown
if edge_count > 0:
    edge_types = preview_df[preview_df["category"] == "edge"]["edge_type"].value_counts()
    html = "<table style='border-collapse:collapse; margin:8px 0;'>"
    html += "<tr style='background:#1a1a2e; color:white;'><th style='padding:6px 12px;'>Edge Type</th><th style='padding:6px 12px;'>Count</th></tr>"
    for et, cnt in edge_types.items():
        html += f"<tr style='border-bottom:1px solid #ddd;'><td style='padding:4px 12px;'><code>{et}</code></td><td style='padding:4px 12px; text-align:center;'>{cnt}</td></tr>"
    html += "</table>"
    display(HTML(html))

Edge Type,Count
ambiguous_query,1
contradictory_data,1
empty_input,1
off_topic,1
prompt_injection,1
multi_doc_confusion,1
numeric_precision,1
excessive_length,1
no_clarification_needed,1
missing_key_fields,1


---
## Section 3 — Collect Agent Responses (Phase 1)

Runs the 3-agent pipeline **sequentially** with rate-limit delays, saving responses to a JSONL file.  
This separates agent calls from evaluator calls so they don't compete for API quota.

In [4]:
PRECOMPUTED_FILE = os.path.join(OUTPUT_DIR, "eval_precomputed_v4.jsonl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def call_agent(agent_name: str, prompt: str) -> str:
    """Invoke a Foundry prompt agent with retry logic."""
    for attempt in range(3):
        try:
            resp = oai.responses.create(
                extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
                input=prompt,
            )
            return resp.output_text
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "RateLimit" in err_str:
                wait = 15 * (attempt + 1)
                print(f"    ⏳ Rate limited, waiting {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
            elif "content_filter" in err_str or "content_management" in err_str:
                return "[CONTENT_FILTERED] The agent declined this request due to content policy."
            else:
                if attempt == 2:
                    return f"[ERROR] Agent call failed: {err_str[:200]}"
                time.sleep(5)
    return "[ERROR] Agent call failed after 3 attempts."


def run_pipeline_for_row(query, context):
    """Run Summarize → Clarify → Report pipeline for one row."""
    ctx = context if context else "No document content provided."

    summary = call_agent(AGENTS["summarizer"],
        f"Based on the following financial document content, answer this question.\n\n"
        f"Document content:\n{ctx}\n\nQuestion: {query}\n\nProvide a clear, concise answer.")

    clarification = call_agent(AGENTS["clarification"],
        f"Review this response to a financial question and identify any gaps.\n\n"
        f"Original question: {query}\nResponse: {summary}")

    return call_agent(AGENTS["reporter"],
        f"Produce a final, polished answer to the following financial question.\n\n"
        f"Question: {query}\n\nInitial analysis:\n{summary}\n\n"
        f"Review feedback:\n{clarification}\n\nSynthesize into a clear, accurate, and complete final answer.")


# Load dataset
input_rows = []
with open(EVAL_DATASET) as f:
    for line in f:
        if line.strip():
            input_rows.append(json.loads(line))

print(f"📞 Phase 1: Calling 3-agent pipeline for {len(input_rows)} rows sequentially...")
print(f"   Each row = 3 agent calls, 2s pause between rows to avoid rate limits.")
print()

t_start = time.time()
precomputed = []
agent_errors = []

for i, row in enumerate(input_rows):
    print(f"  [{i+1:2d}/{len(input_rows)}] {row['query'][:60]}...", end=" ", flush=True)
    try:
        response = run_pipeline_for_row(row["query"], row.get("context", ""))
    except Exception as e:
        response = f"[ERROR] Pipeline failed: {str(e)[:200]}"

    is_error = response.startswith("[ERROR]") or response.startswith("[CONTENT_FILTERED]")
    print("⚠️" if is_error else "✅")
    if is_error:
        agent_errors.append((row["sample_id"], response[:100]))

    precomputed.append({**row, "response": response})

    if i < len(input_rows) - 1:
        time.sleep(2)

# Save precomputed responses
with open(PRECOMPUTED_FILE, "w", encoding="utf-8") as f:
    for row in precomputed:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

elapsed = time.time() - t_start
print(f"\n✅ Phase 1 complete: {len(precomputed)} responses in {elapsed:.0f}s")
print(f"   Saved to: {PRECOMPUTED_FILE}")
if agent_errors:
    print(f"   ⚠️ {len(agent_errors)} rows had errors:")
    for sid, msg in agent_errors:
        print(f"      - {sid}: {msg}")


📞 Phase 1: Calling 3-agent pipeline for 36 rows sequentially...
   Each row = 3 agent calls, 2s pause between rows to avoid rate limits.

  [ 1/36] What is the company name?... ✅
  [ 2/36] What reporting period does this document cover?... ✅
  [ 3/36] What was revenue?... ✅
  [ 4/36] What was net income?... ✅
  [ 5/36] What were operating expenses?... ✅
  [ 6/36] What was operating margin?... ✅
  [ 7/36] What risks were flagged by management?... ✅
  [ 8/36] What FY2026 revenue guidance was provided?... ✅
  [ 9/36] What is the company name?... ✅
  [10/36] What reporting period does this memo cover?... ✅
  [11/36] What was revenue?... ✅
  [12/36] What was net income?... ✅
  [13/36] What was the purpose of the memo?... ✅
  [14/36] What was the operating margin?... ✅
  [15/36] What risks were noted?... ✅
  [16/36] What did management recommend for FY2026?... ✅
  [17/36] What is the company name?... ✅
  [18/36] What reporting period does this workbook cover?... ✅
  [19/36] What was total re

---
## Section 4 — Run Evaluators on Pre-computed Responses (Phase 2)

Runs evaluators in **two batches** on the saved responses (no target function).  
This avoids the 429 rate-limit storm from 7 evaluators × 36 rows hitting the API simultaneously.

- **Batch 1**: Quality evaluators (Groundedness, Relevance, Coherence, Fluency)
- **Batch 2**: Agentic evaluators (TaskAdherence, IntentResolution, ResponseCompleteness)

> ℹ️ The `"Conversation history could not be parsed"` warnings from TaskAdherence/IntentResolution are harmless — they fall back to using the query directly.

In [ ]:
# ── Batch 1: Quality evaluators ──
quality_evaluators = {
    "groundedness":  GroundednessEvaluator(model_config=eval_model_config, credential=credential),
    "relevance":     RelevanceEvaluator(model_config=eval_model_config, credential=credential),
    "coherence":     CoherenceEvaluator(model_config=eval_model_config, credential=credential),
    "fluency":       FluencyEvaluator(model_config=eval_model_config, credential=credential),
}

quality_config = {
    "groundedness": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}", "context": "${data.context}"}
    ),
    "relevance": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}"}
    ),
    "coherence": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}"}
    ),
    "fluency": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}"}
    ),
}

print("🚀 Phase 2a: Running 4 quality evaluators on pre-computed responses...")
print(f"   Data: {PRECOMPUTED_FILE}")
t1 = time.time()

result_quality = evaluate(
    data=PRECOMPUTED_FILE,
    evaluators=quality_evaluators,
    evaluator_config=quality_config,
    evaluation_name=f"financial-v4-quality-{datetime.now().strftime('%Y%m%d-%H%M')}",
    output_path=os.path.join(OUTPUT_DIR, "eval_quality.json"),
)

print(f"✅ Batch 1 done in {time.time()-t1:.0f}s")
for k, v in result_quality.get("metrics", {}).items():
    if isinstance(v, (int, float)):
        print(f"   {k}: {v:.2f}")


In [ ]:
# ── Batch 2: Agentic evaluators (after a 30s cooldown) ──
print("⏳ Pausing 30s before batch 2 to let rate limits reset...")
time.sleep(30)

agentic_evaluators = {
    "task_adherence":        TaskAdherenceEvaluator(model_config=eval_model_config, credential=credential),
    "intent_resolution":     IntentResolutionEvaluator(model_config=eval_model_config, credential=credential),
    "response_completeness": ResponseCompletenessEvaluator(model_config=eval_model_config, credential=credential),
}

agentic_config = {
    "task_adherence": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}"}
    ),
    "intent_resolution": EvaluatorConfig(
        column_mapping={"query": "${data.query}", "response": "${data.response}"}
    ),
    "response_completeness": EvaluatorConfig(
        column_mapping={"response": "${data.response}", "ground_truth": "${data.expected_answer}"}
    ),
}

print("🚀 Phase 2b: Running 3 agentic evaluators...")
t2 = time.time()

result_agentic = evaluate(
    data=PRECOMPUTED_FILE,
    evaluators=agentic_evaluators,
    evaluator_config=agentic_config,
    evaluation_name=f"financial-v4-agentic-{datetime.now().strftime('%Y%m%d-%H%M')}",
    output_path=os.path.join(OUTPUT_DIR, "eval_agentic.json"),
)

print(f"✅ Batch 2 done in {time.time()-t2:.0f}s")
for k, v in result_agentic.get("metrics", {}).items():
    if isinstance(v, (int, float)):
        print(f"   {k}: {v:.2f}")

# ── Merge results ──
print("\n📊 Merging results from both batches...")

result = {
    "metrics": {**result_quality.get("metrics", {}), **result_agentic.get("metrics", {})},
    "rows": [],
}

quality_rows = result_quality.get("rows", [])
agentic_rows = result_agentic.get("rows", [])

for i in range(max(len(quality_rows), len(agentic_rows))):
    merged_row = {}
    if i < len(quality_rows):
        merged_row.update(quality_rows[i])
    if i < len(agentic_rows):
        merged_row.update(agentic_rows[i])
    result["rows"].append(merged_row)

# Save merged results
with open(os.path.join(OUTPUT_DIR, "evaluation_results.json"), "w") as f:
    json.dump(result, f, indent=2, default=str)

print(f"✅ Merged: {len(result['rows'])} rows, {len(result['metrics'])} metrics")


---
## Section 5 — Parse Results

In [ ]:
# Extract metrics and per-row details from merged results
metrics = result.get("metrics", {}) if isinstance(result, dict) else getattr(result, "metrics", {})
rows_data = result.get("rows", []) if isinstance(result, dict) else getattr(result, "rows", [])
rows_df = pd.DataFrame(rows_data)

# Merge with original dataset for category/edge_type info
input_df = pd.read_json(EVAL_DATASET, lines=True)
if len(rows_df) == len(input_df):
    for col in ["sample_id", "category", "edge_type", "document_id", "document_file", "expected_answer"]:
        if col in input_df.columns and col not in rows_df.columns:
            rows_df[col] = input_df[col].values

# Find score column names
eval_names = ["groundedness", "relevance", "coherence", "fluency",
              "task_adherence", "intent_resolution", "response_completeness"]

col_map = {}
for name in eval_names:
    candidates = [c for c in rows_df.columns if name in c.lower()
                  and not any(c.endswith(s) for s in [
                      '_reason','_result','_threshold','_tokens','_model',
                      '_finish_reason','_sample_input','_sample_output',
                      '_prompt_tokens','_completion_tokens','_total_tokens'])]
    if candidates:
        col_map[name] = min(candidates, key=len)

# Report error rows
response_col = next((c for c in rows_df.columns if 'response' in c.lower()
                     and 'completeness' not in c.lower() and c.count('.') <= 1), None)
if response_col:
    error_mask = rows_df[response_col].astype(str).str.contains(
        r'\\[ERROR\\]|\\[CONTENT_FILTERED\\]', na=False, regex=True)
    if error_mask.any():
        print(f"⚠️  {error_mask.sum()} rows had agent errors (flagged in results)")

print(f"Found {len(rows_df)} result rows")
print(f"Score columns mapped: {list(col_map.keys())}")
print(f"\nAggregate metrics:")
for k, v in sorted(metrics.items()):
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.2f}")


---
## Section 6 — Results Dashboard

In [ ]:
def score_color(s, max_val=5):
    pct = (s / max_val) * 100 if max_val > 0 else 0
    if pct >= 80: return "#4CAF50"
    if pct >= 60: return "#FF9800"
    return "#f44336"

# ── KPI cards for all 7 evaluators ──
display(HTML("<h3>📊 Aggregate Scores (Azure AI Evaluation SDK)</h3>"))

labels = {
    "groundedness": "🎯 Groundedness",
    "relevance": "🔗 Relevance",
    "coherence": "🧩 Coherence",
    "fluency": "✍️ Fluency",
    "task_adherence": "📋 Task Adherence",
    "intent_resolution": "🎯 Intent Resolution",
    "response_completeness": "✅ Completeness",
}

cards = "<div style='display:flex; gap:10px; flex-wrap:wrap; margin:16px 0;'>"
for name, label in labels.items():
    col = col_map.get(name)
    if col and col in rows_df.columns:
        avg = pd.to_numeric(rows_df[col], errors="coerce").mean()
        cards += f"""
        <div style='flex:1; min-width:120px; background:{score_color(avg, 5)}; color:white; border-radius:10px; padding:14px; text-align:center;'>
            <div style='font-size:26px; font-weight:bold;'>{avg:.1f}/5</div>
            <div style='font-size:11px; opacity:0.9;'>{label}</div>
        </div>"""
    else:
        cards += f"""
        <div style='flex:1; min-width:120px; background:#999; color:white; border-radius:10px; padding:14px; text-align:center;'>
            <div style='font-size:26px; font-weight:bold;'>N/A</div>
            <div style='font-size:11px; opacity:0.9;'>{label}</div>
        </div>"""
cards += "</div>"
display(HTML(cards))

In [ ]:
# ── Standard vs Edge Case comparison ──
available_evals = [n for n in eval_names if col_map.get(n) in rows_df.columns]

if "category" in rows_df.columns and len(available_evals) > 0:
    std_rows = rows_df[rows_df["category"] == "standard"]
    edge_rows = rows_df[rows_df["category"] == "edge"]

    std_scores = [pd.to_numeric(std_rows[col_map[n]], errors="coerce").mean() for n in available_evals]
    edge_scores = [pd.to_numeric(edge_rows[col_map[n]], errors="coerce").mean() for n in available_evals]

    # Radar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Radar
    angles = np.linspace(0, 2 * np.pi, len(available_evals), endpoint=False).tolist()
    angles += angles[:1]

    ax_radar = fig.add_subplot(121, polar=True)
    std_vals = std_scores + [std_scores[0]]
    edge_vals = edge_scores + [edge_scores[0]]
    ax_radar.fill(angles, std_vals, alpha=0.2, color="#4CAF50")
    ax_radar.plot(angles, std_vals, color="#4CAF50", linewidth=2, label="Standard")
    ax_radar.fill(angles, edge_vals, alpha=0.2, color="#FF9800")
    ax_radar.plot(angles, edge_vals, color="#FF9800", linewidth=2, label="Edge Cases")
    ax_radar.set_xticks(angles[:-1])
    short_labels = [n.replace("_", "\n").title()[:12] for n in available_evals]
    ax_radar.set_xticklabels(short_labels, fontsize=8)
    ax_radar.set_ylim(0, 5)
    ax_radar.set_title("Standard vs Edge Cases", fontsize=13, fontweight="bold", pad=20)
    ax_radar.legend(loc="lower right", fontsize=9)

    # Grouped bar
    axes[1].remove()
    ax_bar = fig.add_subplot(122)
    x = np.arange(len(available_evals))
    w = 0.35
    ax_bar.bar(x - w/2, std_scores, w, label="Standard", color="#4CAF50", edgecolor="white")
    ax_bar.bar(x + w/2, edge_scores, w, label="Edge Cases", color="#FF9800", edgecolor="white")
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels([n.replace("_", "\n") for n in available_evals], fontsize=8, rotation=30, ha="right")
    ax_bar.set_ylim(0, 5.5)
    ax_bar.set_ylabel("Score (1-5)")
    ax_bar.set_title("Score Comparison", fontsize=13, fontweight="bold")
    ax_bar.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "standard_vs_edge.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠️ Cannot compare: missing category column or score columns")

In [ ]:
# ── Heatmap: evaluator scores × question category ──
if "category" in rows_df.columns and len(available_evals) >= 2:
    # Categorize questions
    def categorize(row):
        if row.get("category") == "edge":
            return row.get("edge_type", "edge")
        q = str(row.get("query", "")).lower()
        if "company name" in q: return "company"
        if "period" in q: return "period"
        if "revenue" in q: return "revenue"
        if "net income" in q: return "net_income"
        if "operating" in q and "margin" in q: return "margin"
        if "operating" in q and "expense" in q: return "opex"
        if "risk" in q: return "risks"
        if any(w in q for w in ["guidance","outlook","recommend"]): return "outlook"
        return "other"

    rows_df["q_category"] = rows_df.apply(categorize, axis=1)

    # Build pivot for first 4 evaluators (quality)
    quality_evals = [n for n in ["groundedness","relevance","coherence","fluency"] if n in col_map]
    if quality_evals:
        pivot_data = {}
        for n in quality_evals:
            col = col_map[n]
            pivot_data[n] = pd.to_numeric(rows_df[col], errors="coerce")
        pivot_data["q_category"] = rows_df["q_category"]
        pivot_df = pd.DataFrame(pivot_data)
        pivot = pivot_df.groupby("q_category")[quality_evals].mean()

        fig, ax = plt.subplots(figsize=(10, max(5, len(pivot) * 0.5)))
        im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=5, aspect="auto")
        ax.set_xticks(range(len(quality_evals)))
        ax.set_xticklabels([n.title() for n in quality_evals])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title("🗺️ Quality Scores by Question Category", fontsize=14, fontweight="bold")
        for i in range(len(pivot.index)):
            for j in range(len(quality_evals)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontweight="bold",
                            color="white" if val < 2.5 else "black")
        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "quality_heatmap.png"), dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
# ── Edge case weakness ranking ──
if "edge_type" in rows_df.columns:
    edge_only = rows_df[rows_df["category"] == "edge"].copy()
    if len(edge_only) > 0 and len(available_evals) > 0:
        # Average all available eval scores per edge type
        for n in available_evals:
            edge_only[n] = pd.to_numeric(edge_only[col_map[n]], errors="coerce")
        edge_only["avg_score"] = edge_only[available_evals].mean(axis=1)
        edge_avg = edge_only.groupby("edge_type")["avg_score"].mean().sort_values()

        fig, ax = plt.subplots(figsize=(10, max(4, len(edge_avg) * 0.45)))
        colors = [score_color(s, 5) for s in edge_avg.values]
        bars = ax.barh(edge_avg.index, edge_avg.values, color=colors, edgecolor="white", height=0.6)
        ax.set_xlim(0, 5.5)
        ax.set_xlabel("Avg Score (1-5)")
        ax.set_title("🧪 Edge Cases Ranked by Weakness", fontsize=14, fontweight="bold")
        for bar, val in zip(bars, edge_avg.values):
            ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va="center", fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "edge_case_ranking.png"), dpi=150, bbox_inches="tight")
        plt.show()

---
## Section 7 — Detailed Results Table

In [ ]:
# ── Scrollable detailed results ──
html = """
<style>
  .v4-table { border-collapse:collapse; width:100%; font-size:12px; }
  .v4-table th { background:#1a1a2e; color:white; padding:8px 8px; text-align:left; position:sticky; top:0; }
  .v4-table td { padding:5px 8px; border-bottom:1px solid #eee; vertical-align:top; }
  .v4-table tr:hover { background:#f0f0f0; }
  .badge { display:inline-block; padding:1px 8px; border-radius:10px; color:white; font-weight:bold; font-size:11px; }
</style>
<div style='max-height:500px; overflow-y:auto; border:1px solid #ddd; border-radius:8px;'>
<table class='v4-table'>
<tr><th>ID</th><th>Cat</th><th>Query</th>"""

for name in available_evals:
    short = name[:5].title()
    html += f"<th>{short}</th>"
html += "<th>Response (preview)</th></tr>"

for _, r in rows_df.iterrows():
    sid = r.get("sample_id", "")
    cat = r.get("category", "")
    cat_bg = "#e8f5e9" if cat == "standard" else "#fff3e0"
    query_text = str(r.get("query", r.get("inputs.query", "")))[:80]
    response_text = str(r.get("response", r.get("outputs.response", "")))[:120]

    html += f"<tr style='background:{cat_bg};'>"
    html += f"<td><code>{sid}</code></td>"
    html += f"<td>{cat}</td>"
    html += f"<td>{query_text}</td>"

    for name in available_evals:
        col = col_map[name]
        val = r.get(col, "")
        try:
            val_f = float(val)
            bg = score_color(val_f, 5)
            html += f"<td><span class='badge' style='background:{bg};'>{val_f:.0f}</span></td>"
        except (ValueError, TypeError):
            html += f"<td>{val}</td>"

    html += f"<td style='font-size:11px; max-width:200px;'>{response_text}</td></tr>"

html += "</table></div>"
display(HTML(html))

---
## Section 8 — Export & Final Summary

In [ ]:
# Save detailed results CSV
export_cols = ["sample_id", "category", "edge_type", "document_id", "query"]
export_cols += [col_map[n] for n in available_evals if n in col_map]
export_cols = [c for c in export_cols if c in rows_df.columns]
rows_df[export_cols].to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

# Build summary
summary_lines = []
for name in available_evals:
    col = col_map.get(name)
    if col and col in rows_df.columns:
        avg = pd.to_numeric(rows_df[col], errors="coerce").mean()
        std_avg = pd.to_numeric(rows_df[rows_df.get("category", "") == "standard"][col] if "category" in rows_df.columns else rows_df[col], errors="coerce").mean()
        edge_avg_val = pd.to_numeric(rows_df[rows_df.get("category", "") == "edge"][col] if "category" in rows_df.columns else pd.Series(), errors="coerce").mean()
        summary_lines.append(f"  {name:<25s}  Overall: {avg:.1f}/5  |  Std: {std_avg:.1f}  |  Edge: {edge_avg_val:.1f}")

summary_text = "\n".join(summary_lines)

display(HTML(f"""
<div style='border:2px solid #1a1a2e; border-radius:12px; padding:24px; margin:16px 0; background:linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);'>
  <h2 style='margin-top:0;'>🏦 Financial Agent Evaluation v4 — Final Report</h2>
  <p style='color:#666;'>Run: {timestamp} | Method: Azure AI Evaluation SDK evaluate()</p>
  <p style='color:#666;'>Dataset: {len(rows_df)} rows ({std_count} standard + {edge_count} edge) | Evaluators: {len(available_evals)}</p>
  <hr>

  <h3>📊 Score Summary</h3>
  <pre style='font-size:13px; background:white; padding:12px; border-radius:6px;'>{summary_text}</pre>

  <h3>🔧 How It Works</h3>
  <ul style='font-size:13px;'>
    <li><b>Target function:</b> Each test row → Summarizer → Clarification → Report Generator → final response</li>
    <li><b>evaluate():</b> Single SDK call runs all {len(available_evals)} evaluators across all {len(rows_df)} rows</li>
    <li><b>Quality evaluators:</b> Groundedness, Relevance, Coherence, Fluency (1-5 scale)</li>
    <li><b>Agentic evaluators:</b> TaskAdherence, IntentResolution, ResponseCompleteness (1-5 scale)</li>
  </ul>

  <hr>
  <p style='font-size:12px; color:#888;'>Outputs saved to: {OUTPUT_DIR}/</p>
  <p style='font-size:12px; color:#888;'>Upload to Foundry: {'✅ Enabled' if UPLOAD_TO_FOUNDRY else '⏸️ Disabled (set UPLOAD_TO_FOUNDRY=True)'}</p>
</div>
"""))

print("🎉 Evaluation v4 complete!")